<a href="https://colab.research.google.com/github/kinjalmodi24/bananahacks_26/blob/main/RealPlate_clean.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🍽️ RealPlate
AI restyles the table. Your food stays real, verified pixel by pixel.


In [1]:
# 1. Install
%pip install -q diffusers transformers accelerate gradio scipy

In [2]:
# 2. Load models
import os, numpy as np, torch, gradio as gr
from PIL import Image, ImageDraw, ImageFilter
from scipy.spatial import ConvexHull
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection, SamModel, SamProcessor
from diffusers import AutoPipelineForInpainting

device = "cuda"

dino_id = "IDEA-Research/grounding-dino-tiny"
dino_processor = AutoProcessor.from_pretrained(dino_id)
dino_model = AutoModelForZeroShotObjectDetection.from_pretrained(dino_id).to(device)

sam_id = "facebook/sam-vit-base"
sam_processor = SamProcessor.from_pretrained(sam_id)
sam_model = SamModel.from_pretrained(sam_id).to(device)

pipe = AutoPipelineForInpainting.from_pretrained(
    "diffusers/stable-diffusion-xl-1.0-inpainting-0.1",
    torch_dtype=torch.float16, variant="fp16",
).to(device)

print("models ready")

[transformers] `Siglip2ImageProcessorFast` is deprecated. The `Fast` suffix for image processors has been removed; use `Siglip2ImageProcessor` instead.


preprocessor_config.json:   0%|          | 0.00/457 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.64k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.24k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

...

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

The config attributes {'decay': 0.9999, 'inv_gamma': 1.0, 'min_decay': 0.0, 'optimization_step': 37000, 'power': 0.6666666666666666, 'update_after_step': 0, 'use_ema_warmup': False} were passed to UNet2DConditionModel, but are not expected and will be ignored. Please verify your config.json configuration file.


models ready


In [3]:
# 3. Upload the 5 texture photos (oak.jpg, marble.jpg, slate.jpg, mahogany.jpg, gray.jpg)
from google.colab import files
uploaded = files.upload()
for name in uploaded:
    print("uploaded:", name)

Saving gray.jpg to gray.jpg
Saving mahogany.jpg to mahogany.jpg
Saving slate.jpg to slate.jpg
Saving marble.jpg to marble.jpg
Saving oak.jpg to oak.jpg
uploaded: gray.jpg
uploaded: mahogany.jpg
uploaded: slate.jpg
uploaded: marble.jpg
uploaded: oak.jpg


In [4]:
# 4. Styles and settings
STYLES = {
    "Fine dining (slate)": "overhead flat lay food photography, empty dark charcoal slate tabletop with fine natural stone texture, minimalist, soft even studio light, fine dining, photorealistic, high detail",
    "Cafe brunch (marble)": "overhead flat lay food photography, empty polished white marble tabletop with subtle grey veining, minimalist, bright soft morning light, cafe, photorealistic, high detail",
    "Cozy bistro (oak)": "overhead flat lay food photography, empty aged oak wooden table with visible wood grain, warm soft light, rustic bistro, photorealistic, high detail",
    "Clean studio (gray)": "overhead flat lay food photography, empty smooth gray concrete tabletop, minimalist, bright crisp light, modern studio, photorealistic, high detail",
    "Bar (mahogany)": "overhead flat lay food photography, empty polished dark mahogany wood bar top with soft reflections, minimalist, moody low light, cocktail bar, photorealistic, high detail",
}
NEGATIVE = ("plate, bowl, food, fork, knife, spoon, chopsticks, cup, glass, drink, wine, herbs, garnish, "
            "napkin, props, hands, clutter, blurry, artifacts, text, watermark, logo, signature, letters, words")

STYLE_COLORS = {
    "Fine dining (slate)": (55, 58, 62),
    "Cafe brunch (marble)": (225, 225, 222),
    "Cozy bistro (oak)": (110, 75, 45),
    "Clean studio (gray)": (150, 155, 160),
    "Bar (mahogany)": (75, 35, 25),
}
TEXTURES = {
    "Fine dining (slate)": "/content/slate.jpg",
    "Cafe brunch (marble)": "/content/marble.jpg",
    "Cozy bistro (oak)": "/content/oak.jpg",
    "Clean studio (gray)": "/content/gray.jpg",
    "Bar (mahogany)": "/content/mahogany.jpg",
}
TEX_STRENGTH = 0.5

print("textures found:", [s for s, p in TEXTURES.items() if os.path.exists(p)])

textures found: ['Fine dining (slate)', 'Cafe brunch (marble)', 'Cozy bistro (oak)', 'Clean studio (gray)', 'Bar (mahogany)']


In [5]:
# 5. Pipeline
DISH_PROMPT = "a plate. a bowl. food."

def detect(image, prompt=DISH_PROMPT, threshold=0.3):
    d_in = dino_processor(images=image, text=prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        d_out = dino_model(**d_in)
    det = dino_processor.post_process_grounded_object_detection(
        d_out, input_ids=d_in.input_ids, threshold=threshold, text_threshold=0.25,
        target_sizes=[image.size[::-1]])[0]
    labels = det.get("text_labels", det.get("labels"))
    return [(l, s.item(), b.tolist()) for l, s, b in zip(labels, det["scores"], det["boxes"])]

def box_area(b):
    return (b[2] - b[0]) * (b[3] - b[1])

def build_mask(image):
    """Find the dish, SAM-segment everything on it, merge into one convex hull."""
    dets = detect(image)
    if not dets:
        raise ValueError("No dish detected. Try a clearer photo of the plate.")
    vessels = [d for d in dets if "plate" in d[0] or "bowl" in d[0]]
    main = max(vessels or dets, key=lambda d: box_area(d[2]))
    mx0, my0, mx1, my1 = main[2]
    keep = [d for d in dets
            if mx0 <= (d[2][0] + d[2][2]) / 2 <= mx1 and my0 <= (d[2][1] + d[2][3]) / 2 <= my1]

    s_in = sam_processor(image, input_boxes=[[d[2] for d in keep]], return_tensors="pt").to(device)
    with torch.no_grad():
        s_out = sam_model(**s_in, multimask_output=False)
    masks = sam_processor.post_process_masks(
        s_out.pred_masks.cpu(), s_in["original_sizes"].cpu(), s_in["reshaped_input_sizes"].cpu())[0]
    union = masks[:, 0].any(dim=0).numpy()

    ys, xs = np.nonzero(union)
    pts = np.column_stack([xs, ys])
    hull = ConvexHull(pts)
    hull_img = Image.new("L", image.size, 0)
    ImageDraw.Draw(hull_img).polygon([tuple(map(int, p)) for p in pts[hull.vertices]], fill=255)
    return np.array(hull_img) > 0, keep

def food_coverage_check(mask, keep):
    """Every detected food box must be mostly inside the protected region."""
    ok = True
    for label, score, b in keep:
        if "food" not in label:
            continue
        x0, y0, x1, y1 = [int(round(v)) for v in b]
        ok &= mask[y0:y1, x0:x1].mean() * 100 >= 85
    return ok

def gen_size(w, h, target=1024 * 1024):
    scale = (target / (w * h)) ** 0.5
    return max(64, round(w * scale / 64) * 64), max(64, round(h * scale / 64) * 64)

def add_shadow(background, mask_img, offset=(8, 14), blur=28, strength=0.55):
    shadow = Image.new("L", background.size, 0)
    shadow.paste(mask_img, offset)
    shadow = shadow.filter(ImageFilter.GaussianBlur(blur))
    s = np.array(shadow).astype(np.float32) / 255 * strength
    bg = np.array(background).astype(np.float32) * (1 - s[..., None])
    return Image.fromarray(bg.clip(0, 255).astype(np.uint8))

def verify(original, result, mask):
    diff = np.abs(np.array(result).astype(np.int16) - np.array(original).astype(np.int16)).sum(axis=2)
    print(f"Dish pixels changed: {int((diff[mask] > 0).sum()):,} of {int(mask.sum()):,}")
    return diff

def textured_fill(size, rgb, amount=30, seed=0):
    w, h = size
    rng = np.random.default_rng(seed)
    noise = rng.normal(0, amount, (max(1, h // 16), max(1, w // 16))).astype(np.float32)
    noise = np.array(Image.fromarray(noise, mode="F").resize((w, h), Image.BICUBIC))
    arr = np.array(rgb, dtype=np.float32)[None, None, :] + noise[..., None]
    return Image.fromarray(arr.clip(0, 255).astype("uint8"))

def texture_base(style, size, seed):
    """Real texture photo if we have one, otherwise a noisy colour fill."""
    path = TEXTURES.get(style)
    if path and os.path.exists(path):
        tex = Image.open(path).convert("RGB")
        w, h = size
        scale = max(w / tex.width, h / tex.height)
        tex = tex.resize((round(tex.width * scale) + 1, round(tex.height * scale) + 1), Image.LANCZOS)
        left, top = (tex.width - w) // 2, (tex.height - h) // 2
        return tex.crop((left, top, left + w, top + h)), True
    return textured_fill(size, STYLE_COLORS.get(style, (110, 110, 110)), seed=seed), False

def realplate(image, style="Cozy bistro (oak)", seed=42):
    image = image.convert("RGB")
    if max(image.size) > 1600:
        image.thumbnail((1600, 1600), Image.LANCZOS)

    # 1. Protect the dish
    mask, keep = build_mask(image)
    if not food_coverage_check(mask, keep):
        raise ValueError("Food is not fully inside the protected area. Refusing to generate.")
    mask_img = Image.fromarray((mask * 255).astype("uint8"))

    # 2. Generate the background only
    gw, gh = gen_size(*image.size)
    base, is_tex = texture_base(style, image.size, seed)
    filled = Image.composite(image, base, mask_img)
    bg_mask = Image.fromarray(((~mask) * 255).astype("uint8"))
    gen = pipe(
        prompt=STYLES[style], negative_prompt=NEGATIVE,
        image=filled.resize((gw, gh), Image.LANCZOS),
        mask_image=bg_mask.resize((gw, gh), Image.NEAREST),
        width=gw, height=gh,
        strength=TEX_STRENGTH if is_tex else 1.0,
        guidance_scale=7.5 if is_tex else 10,
        num_inference_steps=30,
        generator=torch.Generator(device).manual_seed(seed),
    ).images[0].resize(image.size, Image.LANCZOS)

    # 3. Paste the original dish back on top + soft shadow
    result = Image.composite(image, add_shadow(gen, mask_img), mask_img)

    # 4. Verify + proof map
    diff = verify(image, result, mask)
    proof = np.zeros((*diff.shape, 3), dtype=np.uint8)
    proof[diff > 0] = [255, 60, 60]
    return result, mask_img, Image.fromarray(proof)

print("pipeline ready")

pipeline ready


In [6]:
# 6. Launch the app
def pad_to_aspect(img, aspect=2, color=(30, 30, 30)):
    w, h = img.size
    if w / h >= aspect:
        new_w, new_h = w, round(w / aspect)
    else:
        new_w, new_h = round(h * aspect), h
    canvas = Image.new("RGB", (new_w, new_h), color)
    canvas.paste(img, ((new_w - w) // 2, (new_h - h) // 2))
    return canvas

def run_realplate(image, style, seed, custom_prompt=""):
    if image is None:
        raise gr.Error("Upload a food photo first.")
    chosen = style
    if custom_prompt and custom_prompt.strip():
        STYLES["Custom"] = custom_prompt.strip()
        chosen = "Custom"
    try:
        result, mask_img, proof = realplate(image, chosen, int(seed))
    except ValueError as e:
        raise gr.Error(str(e))

    mask_arr = np.array(mask_img) > 0
    changed = np.array(proof)[..., 0] > 0
    dish_px = int(mask_arr.sum())
    changed_dish = int((changed & mask_arr).sum())
    bg_pct = changed[~mask_arr].mean() * 100

    if changed_dish == 0:
        status = (f"### ✅ Verified authentic\n"
                  f"**0 of {dish_px:,}** dish pixels changed. "
                  f"The AI generated {bg_pct:.0f}% of the background and never touched the food.")
    else:
        status = f"### ⚠️ {changed_dish:,} dish pixels changed. Do not use this image."

    before = image.convert("RGB").resize(result.size)
    return (pad_to_aspect(before), pad_to_aspect(result)), proof, status

try:
    demo.close()
except Exception:
    pass

with gr.Blocks(title="RealPlate") as demo:
    gr.Markdown("# 🍽️ RealPlate\nAI restyles the table. Your food stays real, verified pixel by pixel.")
    with gr.Row():
        with gr.Column():
            inp = gr.Image(type="pil", label="Your food photo", height=360)
            style = gr.Dropdown(list(STYLES), value="Cozy bistro (oak)", label="Restaurant style")
            custom = gr.Textbox(label="Custom background prompt (optional, overrides the style)", lines=2,
                                placeholder="overhead flat lay food photography, ...")
            seed = gr.Number(value=42, precision=0, label="Variation")
            btn = gr.Button("Generate", variant="primary")
        with gr.Column():
            slider = gr.ImageSlider(label="Before / After", type="pil")
            status = gr.Markdown()
            proof = gr.Image(label="Proof map: red = AI, black = original", type="pil", height=240)
    btn.click(run_realplate, inputs=[inp, style, seed, custom], outputs=[slider, proof, status])

demo.launch(share=True, prevent_thread_lock=True)
print("Public URL:", demo.share_url)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://018dd450b4be5b2eaf.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Public URL: https://018dd450b4be5b2eaf.gradio.live
